# Traceprop inline-precond — Pythia-160M / SST-2 FULL campaign (Part A1)

Run only AFTER the pilot cleared the gate. 200 subsets x 5 seeds. Each seed's JSON + npz +
bootstrap are copied to Drive **as soon as that seed finishes**, and completed seeds are
**restored + skipped** on re-run, so a Colab disconnect loses at most the in-progress seed.

Pinned to commit `25a5624`. Prefer an A100 (~3-5 hrs total).

## Cell 1 — GPU check + mount Drive

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE")
assert any(g in gpu for g in ("A100", "L4")), (
    f"Need an A100 or L4 GPU, got {gpu!r}. Runtime -> Change runtime type -> GPU.")
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Clone repo at the pinned commit + install deps

In [ ]:
%cd /content
!rm -rf Traceprop && git clone -q https://github.com/AmitoVrito/Traceprop.git
%cd /content/Traceprop
!git checkout -q 25a5624
!git log --oneline -1
!pip -q install "datasets>=2.14" "peft>=0.7" pytest
!pip -q install --ignore-requires-python logix-ai
!pip -q install -e . --no-deps
import torch, transformers, peft, logix, scipy, numpy, datasets
print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | logix {logix.__version__} | datasets {datasets.__version__}")
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Cell 3 — Unit tests (stops if anything fails)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "tests/unit/test_lora_logging.py", "-q"])
assert r.returncode == 0, "UNIT TESTS FAILED - do not proceed."
print("unit tests passed")

## Cell 4 — FULL campaign (200 subsets x 5 seeds), per-seed Drive save + resume

Each seed: run exp35 -> run bootstrap -> immediately copy that seed's 3 files to Drive.
Re-running after a disconnect restores finished seeds from Drive and skips them.

In [ ]:
import time, subprocess, sys, os, shutil

os.chdir('/content/Traceprop/experiments')
DRIVE = "/content/drive/MyDrive/traceprop_results/"
os.makedirs(DRIVE, exist_ok=True)
GRID = "1e-6,1e-5,1e-4,1e-3,1e-2,1e-1,1e0,1e1"
SEEDS = [0, 1, 2, 3, 4]

def seed_files(s):
    stem = f"exp35_p160_sst2_seed{s}"
    return [f"results/{stem}.json", f"results/{stem}_raw.npz", f"results/{stem}_bootstrap.json"]

for s in SEEDS:
    j, npz, bj = seed_files(s)
    drive_done = all(os.path.exists(os.path.join(DRIVE, os.path.basename(f))) for f in (j, bj))
    if drive_done:
        for f in seed_files(s):                     # restore so aggregation (cell 5) works
            src = os.path.join(DRIVE, os.path.basename(f))
            if os.path.exists(src):
                shutil.copy(src, f)
        print(f"seed {s}: already in Drive -> restored + skipped")
        continue

    print(f"==== seed {s}: exp35 (200 subsets) ====", flush=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, "exp35_logix_lds.py",
        "--backend", "hf", "--model", "EleutherAI/pythia-160m", "--data", "sst2",
        "--device", "cuda", "--gpu_check", "", "--track", "0",
        "--factored", "--inline_precond", "--tune_logix", "--logix_permodule_damping",
        "--n_train", "1500", "--n_test", "100", "--n_subsets", "200",
        "--subset_frac", "0.5", "--epochs", "3", "--batch", "16",
        "--proj_dim", "512", "--lora_init", "pca", "--seed", str(s),
        "--precond_damping_grid", GRID, "--precond_val_frac", "0.3",
        "--out", j, "--force"])
    assert r.returncode == 0, f"seed {s} exp35 failed"

    r = subprocess.run([sys.executable, "exp35_inlineprecond_bootstrap.py",
                        npz, "--n_boot", "2000", "--seed", "0"])
    assert r.returncode == 0, f"seed {s} bootstrap failed"

    for f in seed_files(s):                          # persist THIS seed before starting the next
        if os.path.exists(f):
            shutil.copy(f, DRIVE)
    print(f"seed {s}: done in {(time.time()-t0)/60:.1f} min -> saved to Drive", flush=True)

print("campaign complete")

## Cell 5 — Aggregate across seeds + apply the decision rule

In [ ]:
import subprocess, sys, os, shutil
os.chdir('/content/Traceprop/experiments')
DRIVE = "/content/drive/MyDrive/traceprop_results/"
for kf in (8, 7):
    subprocess.run([sys.executable, "exp35_fairness_aggregate.py",
                    "--base", "exp35_p160_sst2", "--seeds", "0", "1", "2", "3", "4",
                    "--kfac", str(kf)])
for f in ("exp35_p160_sst2_summary_kfac8.json", "exp35_p160_sst2_summary_kfac7.json"):
    p = os.path.join("results", f)
    if os.path.exists(p):
        shutil.copy(p, DRIVE)
zip_path = shutil.make_archive("/content/p160_campaign_results", "zip", "results")
print("summaries copied to Drive; zip:", zip_path)
from google.colab import files
files.download(zip_path)